# Vortrainierte Netze

Einen großen Bildklassifikator von Grund auf zu trainieren, erfordert einen riesigen Datensatz mit Labels und viel Rechenleistung. Mit einem **vortrainierten Netz** können wir uns das meiste davon sparen: Es ist ein Modell, das bereits auf einem großen Bilddatensatz trainiert wurde und dessen gelernte Filter wir wiederverwenden können. Wir können es entweder direkt zur Klassifikation nutzen oder es per **Transfer Learning** an eine neue Aufgabe anpassen.

Dieses Notebook hat zwei Teile. Zuerst klassifizieren wir mit einem vortrainierten VGG16-Modell ein Bild und sehen, warum die zum Modell passende Vorverarbeitung wichtig ist. Dann machen wir per Transfer Learning aus demselben VGG16 einen kleinen Klassifikator, der Handys von Geldbörsen unterscheidet und mit nur einer Handvoll Bilder trainiert wird.

## Lernziele

Am Ende dieses Notebooks kannst du:

- ein vortrainiertes Keras-Modell (VGG16) laden und damit ein Bild klassifizieren.
- die Vorverarbeitung anwenden, die das Modell verlangt, und erklären, warum sie die Vorhersage verändert.
- ein Transfer-Learning-Modell erstellen, indem du eine Faltungsbasis einfrierst und einen neuen Klassifikationskopf hinzufügst.
- dieses Modell auf einem kleinen eigenen Bilddatensatz trainieren und evaluieren.

## Was ist ein vortrainiertes Netz?

Ein vortrainiertes Netz ist ein Modell, das auf einer groß angelegten Aufgabe zur Bildklassifikation trainiert wurde. Wir können es unverändert zur Bildklassifikation nutzen oder als Ausgangspunkt für Transfer Learning für eine neue, verwandte Aufgabe.

**Vorteile:**

- Wir müssen das Modell nicht von Grund auf trainieren.
- Es lässt sich leicht einbinden.
- Die Inferenz ist schnell.
- Wir können eine sehr gute Leistung erreichen.

**Beispiele:** U-Net (Features in medizinischen Bildern), MobileNet, VGG16/19, ResNet, InceptionV3.

Die meisten vortrainierten Netze in Keras sind auf den [1000 Klassen](https://gist.github.com/yrevar/942d3a0ac09ec9e5eb3a) des [ImageNet](https://www.image-net.org/update-mar-11-2021.php)-Datensatzes trainiert.

In [ ]:
# Bibliotheken importieren
from tensorflow.keras.applications.vgg16 import (
    VGG16,
    decode_predictions,
    preprocess_input,
)
from tensorflow.keras.applications.resnet50 import (
    decode_predictions,
    preprocess_input,
)
from tensorflow.keras.preprocessing import image  # eingebaute Bildklasse von Keras
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import Input, Model

import numpy as np
import matplotlib.pyplot as plt

## Ein Bild mit VGG16 klassifizieren

Laden wir ein Bild, das zu einer der 1000 ImageNet-Klassen gehört (ein paar liegen im Verzeichnis `assets`), verwenden das vortrainierte VGG16-Modell und prüfen das Ergebnis der Klassifikation.

### Bild laden und ansehen

Wir lesen das Bild als PIL-Objekt ein und bringen es auf die Form, die das vortrainierte Modell erwartet (VGG16 wurde auf 224x224-Bildern trainiert).

In [ ]:
img = image.load_img("assets/download.jpeg", target_size=(224, 224))

In [ ]:
type(img)

Wandle das Bild in ein NumPy-Array um (wir können auch den dtype angeben):

In [ ]:
# Bild in ein Array umwandeln, der Datentyp lässt sich auch angeben
img = image.img_to_array(img, dtype="uint8")

In [ ]:
img.shape

Stell das Bild dar:

In [ ]:
# Bild darstellen
plt.imshow(img)

### VGG16-Modell laden

Wir laden VGG16 mit seinem vollständigen Satz an Gewichten. Ohne Argumente lädt Keras das auf ImageNet trainierte Modell herunter, wenn du das zum ersten Mal ausführst.

In [ ]:
# Modell initialisieren
model = VGG16()

In [ ]:
# Zusammenfassung des Modells anzeigen
model.summary()

Erklärung des Ergebnisses: VGG16 hat etwa **138 Millionen Parameter**. Deshalb ist es so aufwendig, ein solches Netz von Grund auf zu trainieren, und deshalb ist es so wertvoll, die vortrainierten Gewichte wiederzuverwenden.

Prüf die Eingabeform, die das Modell erwartet:

In [ ]:
# Vom Modell geforderte Form prüfen
model.input.shape

#### Dimensionen erweitern

Das Modell erwartet einen Batch von Bildern. Deshalb fügen wir vorne eine Batch-Dimension hinzu und machen so aus dem einzelnen 224x224x3-Bild einen Batch mit einem einzigen Bild.

In [ ]:
# Form an die vom Modell geforderte Eingabeform anpassen
img = np.expand_dims(img, axis=0)  # oder img = img.reshape(1,224,224,3)

In [ ]:
img.shape

Sag die Klassenwahrscheinlichkeiten vorher und prüf dann die Form der Vorhersage:

In [ ]:
pred = model.predict(img)

In [ ]:
# Form von pred
pred.shape

Dekodiere den Wahrscheinlichkeitsvektor für die 1000 Klassen in Labels, die Menschen lesen können:

In [ ]:
# Labels dekodieren
decode_predictions(pred)

Erklärung des Ergebnisses: Das Bild zeigt einen Cartoon-Papagei, aber die wahrscheinlichste Vorhersage ist `hair_slide` (etwa 30%), und die ist falsch. Die Pixel wurden unverarbeitet eingegeben und waren daher anders skaliert als beim Training von VGG16. Deshalb müssen wir das Bild genauso vorverarbeiten, wie es beim Training des Modells gemacht wurde.

### Bild vorverarbeiten

Für gute Vorhersagen müssen wir Bilder genau so vorverarbeiten, wie es die Forschenden beim Training des Modells getan haben. Für VGG16 gilt:

> Die Bilder werden von RGB in BGR umgewandelt, dann wird jeder Farbkanal bezogen auf den ImageNet-Datensatz nullzentriert, ohne Skalierung.

Sieh dir die [Dokumentation von `preprocess_input`](https://www.tensorflow.org/api_docs/python/tf/keras/applications/vgg16/preprocess_input) an.

In [ ]:
img = image.load_img("assets/download.jpeg", target_size=(224, 224))
# Im nächsten Durchlauf den Pfad probeweise auf assets/object.png ändern und die Vorhersage des vortrainierten Modells testen
img = np.expand_dims(img, axis=0)  # oder img = img.reshape(1,224,224,3)
img = preprocess_input(
    img
)  # Bild auf dieselbe Weise vorverarbeiten wie beim vortrainierten Modell

Das vorverarbeitete Bild sieht in der Darstellung verzerrt aus, weil die Kanäle verschoben und neu angeordnet wurden:

In [ ]:
plt.imshow(img[0])

#### Erneut vorhersagen

In [ ]:
pred = model.predict(img)

In [ ]:
# Form von pred
pred.shape

In [ ]:
# Labels dekodieren
decode_predictions(pred)

Erklärung des Ergebnisses: Nach der Vorverarbeitung ist die wahrscheinlichste Vorhersage `macaw` (etwa 31%), und die ist richtig: Das Bild zeigt einen Cartoon-Papagei. Geändert hat sich nur, dass wir die zum Modell passende Vorverarbeitung verwendet haben, und das hat aus der falschen eine richtige Vorhersage gemacht.

Lade jetzt das Bild `assets/object.png` und prüf das Ergebnis mit VGG16. Du solltest die vorhergesagte Klasse `cleaver` erhalten. Diskutiere, warum das Modell bei einem Handy `cleaver` vorhersagt.

## Transfer Learning

### Wie Transfer Learning funktioniert

> Transfer Learning besteht darin, Features, die bei einem Problem gelernt wurden, für ein neues, ähnliches Problem zu nutzen. Zum Beispiel können Features aus einem Modell, das gelernt hat, Waschbären zu erkennen, einem Modell, das Tanukis (japanische Waschbären) erkennt, zu einem schnellen Start verhelfen.

**Vorteile:**

- Du verwendest ein vortrainiertes Netz wieder.
- Es spart viel Trainingszeit.
- Es funktioniert mit sehr kleinen Trainingsdatensätzen.

**Vorgehen:**

1. Nimm die Gewichte und die Architektur eines [vortrainierten Netzes](https://keras.io/api/applications/).
2. Lade die Faltungsbasis (alles außer den letzten dichten Schichten).
3. Frier die Basis ein, damit ihre Gewichte unverändert bleiben.
4. Füg obendrauf eine vollständig verbundene dichte Schicht hinzu.
5. Füg eine aufgabenspezifische dichte Ausgabeschicht hinzu.
6. Kompiliere das Modell und trainiere es auf deinen Daten.

```mermaid
flowchart LR
    A[Eingabebild] --> B[VGG16-Faltungsbasis<br/>eingefroren, vortrainiert]
    B --> C[Flatten]
    C --> D[Neue dichte Ausgabeschicht<br/>trainierbar, 2 Klassen]
    D --> E[Handy oder Geldbörse]
```

Nur der neue Kopf (in diesem Notebook die letzte dichte Schicht) wird trainiert. Die eingefrorene Basis behält die allgemeinen visuellen Features, die VGG16 bereits aus ImageNet gelernt hat.

### Daten mit ImageDataGenerator laden

[`ImageDataGenerator`](https://www.tensorflow.org/api_docs/python/tf/keras/preprocessing/image/ImageDataGenerator) erzeugt Batches von Bildtensoren, optional mit Datenaugmentierung in Echtzeit. Das ist ein effizienter Weg, Daten zu laden, die nicht in den Arbeitsspeicher passen, denn der Generator lädt sie per Streaming aus einem Verzeichnis, und trainiert wird in kleinen Batches.

In [ ]:
# Namen der Ordner mit Bildern der Dinge, die du klassifizieren willst
classes = ["phone", "wallet"]

**Datenaugmentierung** wendet zufällige Transformationen auf die Trainingsbilder an (Spiegelungen, Drehungen, Verschiebungen, Skalierung usw.). Sie kann bei begrenzten oder unbalancierten Daten helfen und Overfitting verringern, indem sie den Trainingsdatensatz vergrößert und variiert. Wende sie nur auf die Trainingsdaten an.

In [ ]:
# Datengenerator für Bilder definieren
# Datenaugmentierung: wendet zufällige Verzerrungen und Transformationen auf die Bilder an (nur auf deine Trainingsdaten!).

data_gen = image.ImageDataGenerator(
    # Vorverarbeitungsfunktion definieren, die auf alle Bilder angewendet werden soll
    preprocessing_function=preprocess_input,
    # fill_mode='nearest',
    # rotation_range=20,
    # width_shift_range=0.2,
    # height_shift_range=0.2,
    # horizontal_flip=True,
    # zoom_range=0.2,
    # shear_range=0.2
)

In [ ]:
# -o überschreibt ohne Rückfrage, damit die Zelle ohne Interaktion läuft
!unzip -o -q data.zip

In [ ]:
# Ein Generator, der Batches von X- und y-Arrays zurückgibt
train_data_gen = data_gen.flow_from_directory(
    directory="data/train",
    class_mode="categorical",
    classes=classes,
    batch_size=150,
    target_size=(224, 224),
)

In [ ]:
val_data_gen = data_gen.flow_from_directory(
    directory="data/validation/",
    class_mode="categorical",
    classes=classes,
    batch_size=150,
    target_size=(224, 224),
)

Erklärung des Ergebnisses: Der Generator findet 144 Trainingsbilder und 6 Validierungsbilder, verteilt auf die 2 Klassen. Das ist ein sehr kleiner Datensatz, und genau in solchen Situationen spielt Transfer Learning seine Stärken aus.

In [ ]:
train_data_gen.class_indices

### Transfer-Learning-Modell erstellen

Statt ein CNN von Grund auf zu erstellen, verwenden wir die Faltungsbasis von VGG16 wieder und stimmen sie auf unseren Datensatz fein ab, indem wir die letzten Schichten ersetzen.

#### 1. Faltungsbasis auswählen

Wenn wir das Modell mit `include_top=False` laden, fallen die letzten vollständig verbundenen Schichten von VGG16 (sein ImageNet-Klassifikator) weg. Übrig bleibt nur der Feature-Extraktor aus Faltungsschichten, sodass wir unseren eigenen Klassifikator anhängen können.

In [ ]:
import tensorflow.keras.backend as K

K.clear_session()

# 1. Faltungsbasis / vortrainiertes Netz auswählen
base_model = VGG16(include_top=False)

In [ ]:
base_model.summary()

Erklärung des Ergebnisses: Die Faltungsbasis hat etwa **14,7 Millionen Parameter**, gegenüber 138 Millionen beim vollständigen VGG16, weil die großen dichten Schichten des Klassifikators entfernt wurden.

#### 2. Gewichte einfrieren

Wir frieren die Basis ein, damit ihre vortrainierten Gewichte während des Trainings nicht aktualisiert werden. So bleiben die Features erhalten, die VGG16 bereits gelernt hat, und wir trainieren nur den kleinen neuen Kopf, den wir als Nächstes hinzufügen.

In [ ]:
# 2. Gewichte einfrieren, damit das geladene vortrainierte Modell nicht neu trainiert wird
base_model.trainable = False

In [ ]:
base_model.summary()

Erklärung des Ergebnisses: Nach dem Einfrieren sind alle 14,7 Millionen Parameter nicht mehr trainierbar. Die Basis arbeitet als fester Feature-Extraktor.

#### 3. Eigene dichte Schichten hinzufügen

Wir haben nur 2 Klassen zu klassifizieren (`phone` und `wallet`). Deshalb flachen wir die Ausgabe der Basis ab und fügen als neue Ausgabe eine einzelne dichte Schicht mit 2 Neuronen und einer Softmax-Aktivierung hinzu.

In [ ]:
# 3. Eigenes Modell mit dem vortrainierten Netz als Basismodell erstellen
inputs = Input(shape=(224, 224, 3))

base = base_model(inputs)

# Bei Bedarf lassen sich auch weitere CNN-Schichten hinzufügen

# Nicht vergessen, vor der letzten Schicht abzuflachen
flatten = Flatten()(base)

outputs = Dense(2, activation="softmax")(flatten)

model_tf = Model(inputs, outputs)

In [ ]:
model_tf.summary()

Erklärung des Ergebnisses: Das vollständige Modell hat etwa 14,76 Millionen Parameter, aber nur **50.178** davon sind trainierbar (der neue dichte Kopf). Die 14,7 Millionen eingefrorenen Parameter bleiben unverändert. So wenige Parameter zu trainieren geht schnell und funktioniert sogar mit unserem winzigen Datensatz.

### Kompilieren und trainieren

Wir kompilieren mit dem Adam-Optimierer und der kategorialen Kreuzentropie (zwei per One-Hot Encoding kodierte Klassen) und trainieren dann 10 Epochen lang.

In [ ]:
model_tf.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

model_tf.fit(
    train_data_gen,
    verbose=2,  # wie das Trainingslog ausgegeben werden soll
    epochs=10,
    validation_data=val_data_gen,
)

Erklärung des Ergebnisses: Die Trainings-Accuracy steigt auf fast 99%, und die Validierungs-Accuracy erreicht etwa 0,83 (5 der 6 Validierungsbilder richtig). Bei nur 144 Trainings- und 6 Validierungsbildern ist der Validierungs-Score verrauscht, aber Transfer Learning erreicht trotzdem schnell eine hohe Accuracy, weil die eingefrorene VGG16-Basis bereits reichhaltige visuelle Features liefert.

### Für Vorhersagen nutzen

Wir verwenden das Array `img` von vorhin (den vorverarbeiteten Papagei) wieder, um zu zeigen, wie eine Vorhersage abläuft. Das Modell gibt jetzt für jede der beiden trainierten Klassen, `phone` und `wallet`, eine Wahrscheinlichkeit aus.

In [ ]:
model_tf.predict(img)

In [ ]:
plt.imshow(img[0])

In [ ]:
img.shape

Stell die vorhergesagte Wahrscheinlichkeit für jede Klasse dar:

In [ ]:
plt.bar(x=["phone", "wallet"], height=model_tf.predict(img)[0])

Ordne den Index der höchsten Wahrscheinlichkeit wieder seinem Klassennamen zu:

In [ ]:
# Um die Indexwerte der höchsten Vorhersagewahrscheinlichkeit wieder ihrer jeweiligen Klasse zuzuordnen

pred = model_tf.predict(img)

preds_cls = list(train_data_gen.class_indices.keys())[
    list(train_data_gen.class_indices.values()).index(np.argmax(pred, axis=-1))
]
preds_cls

### Modell speichern

Speichere das trainierte Modell auf der Festplatte, damit du es später neu laden kannst, ohne es neu trainieren zu müssen.

In [ ]:
import os
os.makedirs("models", exist_ok=True)
model_tf.save("models/wallet_phone.h5")

### Aufgabe

- Probier als Basis andere vortrainierte Modelle aus, zum Beispiel ResNet50.
- Wähl einen [Datensatz zur Bildklassifikation](https://www.kaggle.com/datasets?search=image) und erstell dein eigenes CNN (optional): Versuch, es zuerst von Grund auf zu erstellen und dann mit Transfer Learning.
- Große CNNs auf größeren Datensätzen können die Hardware deines Laptops überfordern. Kostenlose Cloud-Ressourcen bieten GPUs (die das Training beschleunigen): [Google Colab](https://colab.research.google.com/) und [Kaggle kernels](https://www.kaggle.com/code) (klick auf `New notebook`; für den GPU-Zugang brauchst du eine verifizierte Telefonnummer).

## Anhang: alle Bilder in Arrays laden

Dies ist eine Alternative zu `ImageDataGenerator`: alle Bilder und Labels direkt in NumPy-Arrays laden. Das ist einfach und reicht für einen kleinen Datensatz, der in den Arbeitsspeicher passt. Anders als beim Generator werden die Daten aber nicht per Streaming geladen, und es gibt keine Datenaugmentierung.

In [ ]:
# Erkunden wir den Datenordner
import os

base_path = "data/train/"

# Definieren wir die Klassen
classes = os.listdir(base_path)

In [ ]:
for class_ in classes:
    print(class_)

In [ ]:
def load_image(base_path):
    """Lädt alle Bilder in X und die Klassen in y"""
    X_list = []
    y_list = []
    classes = os.listdir(base_path)
    for class_ in classes:
        if class_ != ".DS_Store":
            files = os.listdir(base_path + class_)
            for file in files:
                pic = image.load_img(
                    path=base_path + class_ + "/" + f"{file}", target_size=(224, 224)
                )
                numpy_image = np.array(pic)
                processed_image = preprocess_input(numpy_image)
                X_list.append(processed_image)
                y_list.append(class_)

    X = np.array(X_list)
    y = np.array(y_list)

    return X, y, classes

In [ ]:
X, y, classes = load_image(base_path)

In [ ]:
X.shape

In [ ]:
y

In [ ]:
my_dict = {"wallet": 0, "phone": 1}

In [ ]:
# Strings auf binäre Labels abbilden
y = np.vectorize(my_dict.get)(y)
y

## Quellen & weiterführende Literatur

- [**Keras Applications (pretrained models)**](https://keras.io/api/applications/): Der Katalog der vortrainierten Netze, darunter VGG16 und ResNet50.
- [**Transfer learning and fine-tuning (Keras guide)**](https://keras.io/guides/transfer_learning/): Der offizielle Leitfaden zum Einfrieren einer Basis und zum Trainieren eines neuen Kopfes.
- [**Data augmentation (TensorFlow tutorial)**](https://www.tensorflow.org/tutorials/images/data_augmentation): Techniken, um einen Bilddatensatz zu vergrößern und zu variieren.
- [**Transfer learning and fine-tuning (TensorFlow tutorial)**](https://www.tensorflow.org/tutorials/images/transfer_learning): Eine ausführbare End-to-End-Anleitung zu Transfer Learning mit einem echten Bilddatensatz.